## 1. Importação das Bibliotecas

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['font.size'] = 12
plt.style.use('seaborn-v0_8-whitegrid')

## 2. Implementação do Perceptron

In [2]:
class Perceptron:
    """
    Implementação do Perceptron de uma só camada.

    Parâmetros
    ----------
    taxa_aprendizado : float
        Taxa de aprendizado (eta), entre 0.0 e 1.0.
    n_epocas : int
        Número máximo de épocas (passagens pelo conjunto de treino).
    semente : int ou None
        Semente para reprodutibilidade dos pesos iniciais.

    Atributos
    ---------
    pesos_ : ndarray, shape (n_features,)
        Vetor de pesos após o treinamento.
    bias_ : float
        Bias (viés) após o treinamento.
    erros_por_epoca_ : list
        Número de classificações erradas em cada época.
    """

    def __init__(self, taxa_aprendizado=0.1, n_epocas=200, semente=None):
        self.taxa_aprendizado = taxa_aprendizado
        self.n_epocas = n_epocas
        self.semente = semente

    def _funcao_ativacao(self, z):
        """Função de ativação degrau (step function)."""
        return np.where(z >= 0, 1, 0)

    def _soma_ponderada(self, X):
        """Calcula a soma ponderada: z = X·w + b."""
        return np.dot(X, self.pesos_) + self.bias_

    def prever(self, X):
        """Realiza a predição para as entradas X."""
        return self._funcao_ativacao(self._soma_ponderada(X))

    def treinar(self, X, y):
        """
        Treina o perceptron com os dados de entrada X e rótulos y.

        Parâmetros
        ----------
        X : ndarray, shape (n_amostras, n_features)
            Dados de entrada.
        y : ndarray, shape (n_amostras,)
            Rótulos desejados (0 ou 1).

        Retorna
        -------
        self : objeto
        """
        # Inicialização dos pesos e do bias (inicializado como 1.0)
        rng = np.random.default_rng(self.semente)
        self.pesos_ = rng.normal(loc=0.0, scale=0.01, size=X.shape[1])
        self.bias_ = 1.0  # Conforme solicitado, o bias começa em 1
        self.erros_por_epoca_ = []

        print(f"Pesos iniciais: {self.pesos_}")
        print(f"Bias inicial:   {self.bias_}")
        print(f"Taxa de aprendizado: {self.taxa_aprendizado}")
        print(f"Épocas máximas: {self.n_epocas}")
        print("-" * 40)

        for epoca in range(1, self.n_epocas + 1):
            erros = 0

            for xi, yi in zip(X, y):
                # Predição
                y_pred = self.prever(xi)

                # Cálculo do erro
                erro = yi - y_pred

                # Atualização dos pesos e bias
                if erro != 0:
                    self.pesos_ += self.taxa_aprendizado * erro * xi
                    self.bias_ += self.taxa_aprendizado * erro
                    erros += 1

            self.erros_por_epoca_.append(erros)

            # Mostrar progresso a cada 10 épocas ou quando convergir
            if epoca % 10 == 0 or erros == 0:
                print(f"Época {epoca:3d} | Erros: {erros}")

            # Convergência: nenhum erro
            if erros == 0:
                print(f"\nConvergiu na época {epoca}!")
                break
        else:
            print(f"\nNão convergiu em {self.n_epocas} épocas.")

        print(f"\nPesos finais: {self.pesos_}")
        print(f"Bias final:   {self.bias_}")

        return self

    def acuracia(self, X, y):
        """Calcula a acurácia do modelo."""
        y_pred = self.prever(X)
        return np.mean(y_pred == y) * 100

---

## 3. Experimento: Classificação com Dataset Iris

In [3]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

# Carregar o dataset Iris
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

# Selecionar apenas Setosa (0) e Versicolor (1)
# Usar apenas 2 features: comprimento e largura da sépala
mask = y_iris < 2
X_iris_2 = X_iris[mask, :2]  # sepal length e sepal width
y_iris_2 = y_iris[mask]

print(f"Amostras: {X_iris_2.shape[0]}")
print(f"Features: {iris.feature_names[:2]}")
print(f"Classes:  {iris.target_names[:2]}")
print(f"Distribuição - Setosa: {(y_iris_2 == 0).sum()} | Versicolor: {(y_iris_2 == 1).sum()}")

Amostras: 100
Features: ['sepal length (cm)', 'sepal width (cm)']
Classes:  ['setosa' 'versicolor']
Distribuição - Setosa: 50 | Versicolor: 50


In [4]:
# Dividir em treino e teste
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X_iris_2, y_iris_2, test_size=0.3, random_state=42, stratify=y_iris_2
)

print(f"Amostras de treino: {X_treino.shape[0]}")
print(f"Amostras de teste:  {X_teste.shape[0]}")

Amostras de treino: 70
Amostras de teste:  30


In [5]:
p_iris = Perceptron(taxa_aprendizado=0.01, n_epocas=200, semente=42)
p_iris.treinar(X_treino, y_treino)

acc_treino = p_iris.acuracia(X_treino, y_treino)
acc_teste  = p_iris.acuracia(X_teste, y_teste)

print(f"Épocas até convergir: {len(p_iris.erros_por_epoca_)}")
print(f"Acurácia no Treino:   {acc_treino:.1f}%")
print(f"Acurácia no Teste:    {acc_teste:.1f}%")

Pesos iniciais: [ 0.00304717 -0.01039984]
Bias inicial:   1.0
Taxa de aprendizado: 0.01
Épocas máximas: 200
----------------------------------------
Época  10 | Erros: 7
Época  20 | Erros: 4
Época  30 | Erros: 6
Época  40 | Erros: 6
Época  50 | Erros: 4
Época  60 | Erros: 4
Época  70 | Erros: 2
Época  80 | Erros: 3
Época  90 | Erros: 3
Época 100 | Erros: 3
Época 110 | Erros: 3
Época 120 | Erros: 3
Época 130 | Erros: 2
Época 140 | Erros: 4
Época 150 | Erros: 2
Época 160 | Erros: 4
Época 170 | Erros: 3
Época 180 | Erros: 2
Época 190 | Erros: 2
Época 200 | Erros: 2

Não convergiu em 200 épocas.

Pesos finais: [ 0.53604717 -1.03239984]
Bias final:   0.25999999999999934
Épocas até convergir: 200
Acurácia no Treino:   98.6%
Acurácia no Teste:    96.7%
